# RQ2 runs (Kaggle, T4 x2): consistency and precedents

Runs the full system (fine-tuned 3B verifier), B3 and B1 on the 32 RQ2 consistency cases
(`data/cases/rq2/`: 4 equivalence groups of 4 submissions and 8 minimal pairs on the dev
assignment D1), and the precedent runs: the store fills with the system's own decisions in
three case orders, plus one run with a planted wrong precedent (`docs/DECISIONS.md`
section 16). The code and prompts are the frozen version; only the RQ2 settings are new.

`RUNS` holds the second session (two repeats of the run without precedents, to measure
run-to-run noise; a wrong precedent on a target the system gets right; a correct precedent
on the faulty test). The first session's runs are listed in `FIRST_SESSION`.

**Notebook settings:** Accelerator *GPU T4 x2*, Internet *on*, about 1 hour for `RUNS`
(about 2 hours for `FIRST_SESSION`).
**Input** (*Add Input*): the output of the `verifier_ft` notebook version that trained the
verifier (`verifier_model/`, or its adapter, which is merged here).

**Output:** `rq2_outputs.zip` with every run directory (config, events, predictions,
proposals, metrics; the consistency metrics are in `metrics.json` under `consistency`),
the server logs, commit and `pip freeze`. The cases are public dev data, so all of it can
be committed under `results/rq2/`.


In [ ]:
REPO_URL = "https://github.com/JagadishReddyK7/regrade-appeals-agent.git"
BRANCH = "main"
VLLM_SPEC = "vllm==0.32.0"  # as in the dev and test runs
WORKDIR = "/kaggle/temp/regrade-appeals-agent"
OUT = "/kaggle/working/rq2_outputs"
SERVED_PATH = "/kaggle/temp/verifier-ft"  # configs/rq2_pipeline*.yaml serve this path
MERGE_DIR = "/kaggle/temp/verifier_model"  # used only if the input has just the adapter
FIRST_SESSION = [  # done on 2026-11-16 (results/rq2/); add to RUNS to repeat them
    "rq2_pipeline",  # full system, empty precedent store
    "rq2_pipeline_precedents_s1",  # precedent runs: the store fills with own decisions
    "rq2_pipeline_precedents_s2",
    "rq2_pipeline_precedents_s3",
    "rq2_pipeline_planted",  # one wrong precedent (faulty test) from the start
    "rq2_agent",  # B3
    "rq2_b1",  # B1
]
RUNS = [  # second session, in priority order; all need the 7B and the fine-tuned 3B
    "rq2_pipeline_r2",  # repeats of rq2_pipeline: run-to-run noise
    "rq2_pipeline_r3",
    "rq2_pipeline_planted_recursion",  # wrong precedent on a target the system gets right
    "rq2_pipeline_helpful",  # correct precedent on the faulty test
]

In [ ]:
# Clone. The token (if any) is removed from the git remote afterwards.
import glob
import json
import os
import shutil
import subprocess

url = REPO_URL
try:
    from kaggle_secrets import UserSecretsClient
    url = REPO_URL.replace("https://", f"https://{UserSecretsClient().get_secret('GITHUB_TOKEN')}@")
except Exception:
    pass  # public repo, or no secret configured
if not os.path.exists(WORKDIR):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, WORKDIR], check=True)
subprocess.run(["git", "-C", WORKDIR, "remote", "set-url", "origin", REPO_URL], check=True)
os.chdir(WORKDIR)
os.makedirs(OUT, exist_ok=True)
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # same GPU numbering as nvidia-smi
!git log -1 --oneline

## Fine-tuned verifier and install


In [ ]:
# The fine-tuned verifier: the merged model from the verifier_ft output, else its adapter.
merged = sorted(glob.glob("/kaggle/input/**/verifier_model/config.json", recursive=True))
adapters = sorted(glob.glob("/kaggle/input/**/runs/verifier_train-*/adapter/adapter_config.json",
                            recursive=True))
NEED_MERGE = not merged
if merged:
    model_dir = os.path.dirname(merged[-1])
    print("using the merged verifier from", model_dir)
else:
    assert adapters, "no verifier_model/ or adapter in the inputs: add the verifier_ft output"
    print("only the adapter found: it is merged after the install, from", adapters[-1])

In [ ]:
# Install. A merge (only if needed) runs before vLLM, with the training libraries, as in
# notebooks/verifier_ft.ipynb (torchao uninstalled: peft refuses the merge next to it).
def run(*args):
    done = subprocess.run(["python", "-m", *args], capture_output=True, text=True)
    print(done.stdout[-4000:], done.stderr[-4000:])
    return done.returncode


if NEED_MERGE:
    subprocess.run(["pip", "install", "-q", "-e", ".[gpu]"], check=True)
    subprocess.run(["pip", "uninstall", "-y", "-q", "torchao"], check=True)
    source = os.path.dirname(os.path.dirname(adapters[-1]))
    target = f"runs/{os.path.basename(source)}"
    shutil.copytree(source, target, dirs_exist_ok=True)
    assert run("verifier.train", "--config", "configs/verifier_train.yaml",
               "--merge-only", target) == 0, "merge failed: see the output above"
    if os.path.exists(MERGE_DIR):
        shutil.rmtree(MERGE_DIR)
    shutil.move(f"{target}/merged", MERGE_DIR)
    shutil.rmtree(target)  # not part of the test runs
    model_dir = MERGE_DIR
if os.path.lexists(SERVED_PATH):
    os.remove(SERVED_PATH)
os.symlink(model_dir, SERVED_PATH)
!pip install -q -e .
!pip install -q "{VLLM_SPEC}"
!pip uninstall -y -q torchaudio
!python -c "import torch, vllm; print('torch', torch.__version__, 'vllm', vllm.__version__)"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Servers and helpers


In [ ]:
import signal
import time


def gpu_used_mb(gpu):
    query = ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits",
             "-i", str(gpu)]
    return int(subprocess.run(query, capture_output=True, text=True).stdout.strip() or 0)


def stop_server(stem, gpu):
    # Stop a vLLM server and its workers (one process group); wait until the GPU is free.
    if os.path.exists(f"{stem}.pid"):
        try:
            os.killpg(int(open(f"{stem}.pid").read()), signal.SIGTERM)
        except ProcessLookupError:
            pass
        os.remove(f"{stem}.pid")
    for _ in range(60):
        if gpu_used_mb(gpu) < 1000:
            return True
        time.sleep(2)
    print(f"GPU {gpu} is still busy after stopping {stem}")
    return False


def keep_log(stem, label):
    if os.path.exists(f"{stem}.log"):
        shutil.copy(f"{stem}.log", f"{OUT}/{label}.log")


DONE, FAILED = [], []


def save(run_dir):
    # The whole run directory to OUT, right after each run (public dev data).
    name = os.path.basename(run_dir)
    shutil.copytree(run_dir, f"{OUT}/runs/{name}", dirs_exist_ok=True)


def evaluate(config):
    before = set(glob.glob(f"runs/{config}-*"))
    code = run("eval.run", "--config", f"configs/{config}.yaml")
    new = sorted(set(glob.glob(f"runs/{config}-*")) - before)
    for run_dir in new:
        save(run_dir)
    (DONE if code == 0 and new else FAILED).append(config)
    print(config, "done" if code == 0 else "FAILED")


MAIN_READY = run("llm.serve", "--config", "configs/rq2_pipeline.yaml") == 0
FT_READY = run("llm.serve", "--config", "configs/rq2_pipeline.yaml", "--verifier") == 0
SMOKE_OK = MAIN_READY and run("llm.smoke", "--config", "configs/rq2_pipeline.yaml") == 0
print({"7B": MAIN_READY, "fine-tuned 3B": FT_READY, "smoke": SMOKE_OK})

## Runs
About 10-15 minutes per pipeline run on 32 cases.


In [ ]:
for config in RUNS:
    if not SMOKE_OK or not FT_READY:
        print("skipped (server not ready):", config)
        FAILED.append(config)
        continue
    evaluate(config)

## Consistency summary


In [ ]:
# Consistency per run (eval/metrics.py, compute_consistency).
for path in sorted(glob.glob("runs/rq2_*-*/metrics.json")):
    m = json.load(open(path))
    c, o = m["consistency"], m["overall"]
    print(m["run_id"].rsplit("-", 3)[0].ljust(30),
          "acc", o["exact_decision_accuracy"], "| groups", c["group_consistency"],
          c["group_split"], "| gap", c["group_mark_gap"], "| pair flips", c["pair_flip_rate"],
          "| copy", c["precedent_copy_rate"], "| planted copy", c.get("planted_copy_rate"),
          "follow", c.get("planted_follow_rate"))

## Save outputs and stop the servers


In [ ]:
keep_log("vllm", "vllm_7b")
keep_log("vllm_verifier", "vllm_ft3b")
json.dump({"done": DONE, "failed_or_skipped": FAILED, "verifier_merged_here": NEED_MERGE},
          open(f"{OUT}/run_summary.json", "w"), indent=2)
!pip freeze > {OUT}/pip_freeze.txt
!git rev-parse HEAD > {OUT}/commit.txt
shutil.make_archive("/kaggle/working/rq2_outputs", "zip", OUT)
stop_server("vllm", gpu=0)
stop_server("vllm_verifier", gpu=1)
print("done")